# Task 3 — Notebook 03: Maps, time series, state×crop table, run bundle

**Inputs:** `smap_anomaly_2019.parquet`, CDL spatial metadata, Corn Belt state boundaries.

**Figures:** 4-panel z-maps (ISO weeks 14, 18, 22, 27), mean z time series, fraction of weeks with z > 1.5.

**Table:** `artifacts/tables/task3/task3__anomaly_stats_by_state_crop__*.csv`


In [ ]:
import json
import sys
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import load_cdl_spatial_metadata
from src.modeling.task3_aggregate import attach_state_name, state_crop_anomaly_summary
from src.viz.rotation_maps import load_cornbelt_state_boundaries_5070
from src.viz.task3_maps import fill_raster, plot_extent_from_metadata, plot_z_map

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

meta = load_cdl_spatial_metadata(REPO_ROOT)
h, w = int(meta["height"]), int(meta["width"])
extent = plot_extent_from_metadata(meta)
states = load_cornbelt_state_boundaries_5070(REPO_ROOT)

anom_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "smap_anomaly_2019.parquet"
if not anom_pq.is_file():
    raise FileNotFoundError("Run notebook 02 first")
anom = pd.read_parquet(anom_pq)
anom = attach_state_name(REPO_ROOT, anom)

fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
tbl_dir = REPO_ROOT / cfg["output"]["tables_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)
tbl_dir.mkdir(parents=True, exist_ok=True)
date_s = date.today().strftime("%Y%m%d")

targets = [14, 18, 22, 27]
fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=120)
axes = axes.ravel()
for ax, tw in zip(axes, targets):
    sub = anom.loc[anom["iso_week"] == tw].drop_duplicates(subset=["iy", "ix"])
    if sub.empty:
        ax.set_axis_off()
        continue
    g = fill_raster(h, w, sub["iy"].to_numpy(), sub["ix"].to_numpy(), sub["z_score"].to_numpy())
    ax.imshow(g, extent=list(extent), origin="upper", interpolation="nearest", cmap="RdBu_r", vmin=-3, vmax=3)
    if states is not None and not states.empty:
        states.boundary.plot(ax=ax, color="#222", linewidth=0.6)
    d0 = sub["date"].iloc[0]
    ax.set_title(f"ISO week {tw} ({d0})")
    ax.set_axis_off()
fig.suptitle("2019 SMAP soil moisture z-score (vs 2015–2021 weekly climatology)", fontsize=12)
fig.tight_layout()
p4 = fig_dir / f"task3__anomaly_map_4panel__{date_s}.png"
fig.savefig(p4, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", p4.relative_to(REPO_ROOT))

ts = anom.groupby("date", as_index=False).agg(mean_z=("z_score", "mean"), std_z=("z_score", "std"), n=("z_score", "count"))
fig2, ax2 = plt.subplots(figsize=(10, 4))
x = np.arange(len(ts))
ax2.plot(x, ts["mean_z"], color="#2980b9", lw=2)
ax2.fill_between(x, ts["mean_z"] - ts["std_z"], ts["mean_z"] + ts["std_z"], alpha=0.25, color="#2980b9")
ax2.axhline(0, color="k", lw=0.6)
ax2.axhline(1, color="g", ls="--", lw=0.7)
ax2.axhline(-1, color="orange", ls="--", lw=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(ts["date"], rotation=45, ha="right", fontsize=7)
ax2.set_ylabel("mean z (rotation-eligible pixels)")
ax2.set_title("2019 event window — area-mean soil moisture anomaly")
fig2.tight_layout()
pts = fig_dir / f"task3__anomaly_timeseries_cropland__{date_s}.png"
fig2.savefig(pts, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", pts.relative_to(REPO_ROOT))

cnt = anom.assign(hi=anom["z_score"] > 1.5).groupby(["iy", "ix"])["hi"].mean().reset_index(name="frac_weeks_gt_1p5")
g3 = fill_raster(h, w, cnt["iy"].to_numpy(), cnt["ix"].to_numpy(), cnt["frac_weeks_gt_1p5"].to_numpy())
fig3, ax3 = plot_z_map(g3, extent, title="Fraction of event weeks with z > 1.5", vmin=0, vmax=1, cmap="Blues")
fig3.savefig(fig_dir / f"task3__flood_duration_fraction__{date_s}.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved", (fig_dir / f"task3__flood_duration_fraction__{date_s}.png").relative_to(REPO_ROOT))

tab = state_crop_anomaly_summary(anom)
csv_path = tbl_dir / f"task3__anomaly_stats_by_state_crop__{date_s}.csv"
tab.to_csv(csv_path, index=False)
print("Wrote", csv_path.relative_to(REPO_ROOT))

run_id = uuid.uuid4().hex[:8]
run_dir = REPO_ROOT / "artifacts" / "logs" / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    "task": "task3_soil_moisture",
    "run_id": run_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config_path": "configs/task3_soil_moisture.yaml",
    "outputs": {
        "smap_anomaly_2019": str(anom_pq.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_4panel": str(p4.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_timeseries": str(pts.relative_to(REPO_ROOT)).replace("\\", "/"),
        "figures_duration": str((fig_dir / f"task3__flood_duration_fraction__{date_s}.png").relative_to(REPO_ROOT)).replace("\\", "/"),
        "table_state_crop": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
    },
}
(run_dir / "run_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Wrote", (run_dir / "run_bundle.json").relative_to(REPO_ROOT))
